In [1]:
import pyspark
import pandas as pd
from pyspark.sql import types
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 16:46:44 WARN Utils: Your hostname, adbhut-HP, resolves to a loopback address: 127.0.1.1; using 192.168.31.12 instead (on interface wlo1)
26/03/08 16:46:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 16:46:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read \
    .option("header", "true") \
    .csv("fhvhv_tripdata_2021-01.csv")

In [4]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [5]:
df_pandas = pd.read_csv('head.csv')

In [6]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [7]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [8]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [9]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("fhvhv_tripdata_2021-01.csv")

In [10]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [11]:
df.rdd.getNumPartitions()

22

In [12]:
df = df.repartition(24)

In [14]:
df.write.parquet('fhvhv/2021/01/', mode='overwrite')

26/03/08 16:47:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/08 16:47:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/08 16:47:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/08 16:47:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/08 16:47:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/08 16:47:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/08 16:47:36 WARN MemoryManager: Total allocation exceeds 95.

In [15]:
df = spark.read.parquet('fhvhv/2021/01/')

In [16]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [17]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num  == "HV0003") \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 12:04:59|2021-01-02 12:24:52|         146|          97|
|2021-01-01 21:35:15|2021-01-01 21:47:42|         125|          48|
|2021-01-01 15:20:24|2021-01-01 15:36:18|         235|          60|
|2021-01-01 15:16:55|2021-01-01 15:26:45|         145|         146|
|2021-01-01 01:20:16|2021-01-01 01:25:11|         143|         142|
|2021-01-01 09:11:18|2021-01-01 09:21:39|         206|         115|
|2021-01-01 14:57:07|2021-01-01 15:20:42|          67|         232|
|2021-01-02 08:50:10|2021-01-02 09:39:13|         177|          42|
|2021-01-02 13:02:36|2021-01-02 13:12:47|          87|         144|
|2021-01-01 00:39:11|2021-01-01 00:43:55|         121|         131|
|2021-01-01 04:28:12|2021-01-01 04:38:12|           3|         182|
|2021-01-01 00:51:26|2021-01-01 01:01:06|       

In [18]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-02|  2021-01-02|         146|          97|
| 2021-01-01|  2021-01-01|          97|         255|
| 2021-01-01|  2021-01-01|         125|          48|
| 2021-01-01|  2021-01-01|         235|          60|
| 2021-01-01|  2021-01-01|          73|         265|
| 2021-01-01|  2021-01-01|         145|         146|
| 2021-01-01|  2021-01-01|         143|         142|
| 2021-01-01|  2021-01-01|         206|         115|
| 2021-01-01|  2021-01-01|         132|         265|
| 2021-01-01|  2021-01-01|          67|         232|
| 2021-01-02|  2021-01-02|          39|         132|
| 2021-01-02|  2021-01-02|         177|          42|
| 2021-01-02|  2021-01-02|          87|         144|
| 2021-01-01|  2021-01-01|         141|         160|
| 2021-01-01|  2021-01-01|         121|         131|
| 2021-01-01|  2021-01-01|           3|       

In [19]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [20]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [21]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

[Stage 8:>                                                          (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/acc| 2021-01-02|  2021-01-02|         146|          97|
|  e/9ce| 2021-01-01|  2021-01-01|          97|         255|
|  e/b3b| 2021-01-01|  2021-01-01|         125|          48|
|  e/a39| 2021-01-01|  2021-01-01|         235|          60|
|  e/9ce| 2021-01-01|  2021-01-01|          73|         265|
|  e/b3c| 2021-01-01|  2021-01-01|         145|         146|
|  s/acd| 2021-01-01|  2021-01-01|         143|         142|
|  a/b37| 2021-01-01|  2021-01-01|         206|         115|
|  e/9ce| 2021-01-01|  2021-01-01|         132|         265|
|  e/b3e| 2021-01-01|  2021-01-01|          67|         232|
|  e/9ce| 2021-01-02|  2021-01-02|          39|         132|
|  e/a39| 2021-01-02|  2021-01-02|         177|          42|
|  a/b43| 2021-01-02|  2021-01-02|          87|         144|
|  e/9ce| 2021-01-01|  2

In [22]:
spark.stop()